Sample of code- (needs original data files to run). Currently working on cleaning it up and adding to github.

Typically streamlit needs a .py file to run. Showing here in .ipynb format just for readability

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# data cleaning
# working on adding a google drive API to read in data
nhanes = pd.read_csv('2017-2020_Nhanes_NASEM7_Summaries.csv')
pfas_all = pd.read_csv('GenX_2017-2023_PFAS_Data.csv')
demographics_2017_19 = pd.read_csv('QX_2017-2019_Data.csv')
demographics_2020_21 = pd.read_csv('QX_2020-2021_Data.csv')
demographics_2023 = pd.read_csv('QX_2023_Data.csv')

demographics_2017_19['community'] = demographics_2017_19['community'].replace({
    "1": "LCFR",
    '2': 'Fayetteville',
	'3': 'Pittsboro'
})
demographics_2020_21['community'] = demographics_2020_21['community'].replace({
    "1": "LCFR",
    '2': 'Fayetteville',
	'3': 'Pittsboro'
})
demographics_2023['community'] = demographics_2023['community'].replace({
    "1": "LCFR",
    '2': 'Fayetteville',
	'3': 'Pittsboro'
})

demographics = pd.concat([demographics_2017_19, demographics_2020_21, demographics_2023])
demographics['unique_id'] = demographics['id'].astype(str) + '_' + demographics['community'].astype(str)
demographics= demographics.drop_duplicates('unique_id')
pfas = pfas_all.copy()

pfas['unique_id'] = pfas['id'].astype(str) + '_' + pfas['study_loc'].astype(str)

joined = pd.merge(demographics, pfas, on=["unique_id"], how="right")
joined = joined.drop_duplicates()
nhanes = pd.DataFrame(nhanes)
joined_filter = joined[joined['PFAS'].isin(["PFOS", "PFOA", "PFHxS", "PFNA", "PFDA", "PFUnDA", "MeFOSAA", "Nafion byproduct 2", "PFO5DoA"])]

complete = joined_filter.copy()
complete['study_loc'] = complete['study_loc'].replace({
    "LCFR": "Wilmington"
})

In [ ]:
# replacements using code book
complete['age_cat'] = complete['age_cat'].astype('category')
complete['years_lived_cat'] = complete['years_lived_cat'].astype('category')
complete['community'] = complete['community'].astype('category')
complete['gender'] = complete['gender'].astype('category')
complete['hispanic'] = complete['hispanic'].astype('category')
complete['study_year'] = complete['study_year'].astype('category')
complete['PFAS'] = complete['PFAS'].astype("category")

complete['hispanic'] = complete['hispanic'].replace({
    "0": "Non-Hispanic",
    "1": "Hispanic"
})
complete['years_lived_cat'] = complete['years_lived_cat'].astype(str).replace({
    "1": "1-5",
    "2": "6-10",
    "3": "11-15",
    "4": "16-20",
    "5": "21-25",
    "6": "26-30",
    "7": "31-35",
    "8": "36-40",
    "9": "41-45",
    "10": "46-50",
    "11": "51-55",
    "12": "56-60",
    "13": ">60 years"
})
complete['age_cat'] = complete['age_cat'].astype(str).replace({
    "1": "6-17", "2": "6-17", "3": "18-50", "4": "18-50",
    "5": "51-70", "6": ">70"
})
complete['gender'] = complete['gender'].astype(str).replace({
    "1.0": "Male",
    "2.0": "Female",
    "3.0": "Female" # rolling up as discussed in meeting
})

In [ ]:
# manually grouping chemicals based on known ranges for plotting later
pfas_order = ["PFOS", "PFOA", "PFO5DoA", "PFHxS", "PFNA", "PFDA", "PFUnDA", "MeFOSAA", "Nafion byproduct 2"]
predefined_large_order = ["PFOS", "PFOA", "PFO5DoA"]
predefined_small_order = ["PFHxS", "MeFOSAA", "Nafion byproduct 2"]
predefined_medium_order = ["PFNA", "PFDA", "PFUnDA"]

complete = complete.copy()

gender_order = ['Female', 'Male']

group_colors = {'Pittsboro':'#008473', 'Fayetteville':'#D14905', 'Wilmington':'#4156A1'}
group_colors_pfas = {'PFOA':'#FFA07A', 'PFOS':'#B3C774', 'PFNA':'#FF8C42', 'PFO5DoA':'#66CDAA', 'PFHxS':'#A5B3E0',
                     'PFDA':'#8093E1', 'PFUnDA':'#52B8A7', 'MeFOSAA':'#9FC08B', 'Nafion byproduct 2':'#FFB6C1'}

In [ ]:
loc_order = ["Pittsboro", "Fayetteville", "Wilmington"]

complete.rename(columns={'study_loc': 'Location', 'study_year': 'Year'}, inplace=True)
age_order = ["6-17", "18-50", "51-70", ">70"]
year_cat_order = ["1-5", "6-10", "11-15", "16-20", "21-25", "26-30", "31-35",
              "36-40", "41-45","46-50", "51-55", "56-60", ">60 years"]

# setting style for boxplots later
plt.rcParams['font.family'] = 'Lato'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

sns.set_style("whitegrid")